# Session 10: Making a small model tell the truth

This first pass builds the model and a real next-character training loop. It is intentionally small and self-contained so later experiments can inspect the same computations without downloading data. The toy validation split is only a pipeline check, not evidence of generalization.

The model is a decoder-only Transformer: character and position embeddings, two causal self-attention blocks, and a next-character prediction head. Each training target is the input sequence shifted one character to the right.

In [14]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1337)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"PyTorch {torch.__version__}; device: {device}")

PyTorch 2.5.1; device: mps


## Data and character vocabulary

The short, original text below is repeated to make enough next-character windows for a tiny training run. The split is by position in the text; because the corpus repeats, its validation loss must not be interpreted as a generalization result.

In [16]:
passage = """A careful model learns one character at a time. A useful training loop measures the loss, checks the gradients, and updates each weight.
When the input is a short sequence, the target is the same sequence moved one step forward. The model can use the past, but never the future.
Small experiments make hidden details visible. We print tensor shapes, compare a numerical gradient, and keep track of every example in a batch.
A falling loss is a clue, not a proof. We ask what changed, how much computation was used, and whether the measurements support the conclusion.
"""
text = passage * 24
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {char: index for index, char in enumerate(chars)}
itos = {index: char for char, index in stoi.items()}
encode = lambda value: [stoi[char] for char in value]
decode = lambda values: "".join(itos[int(index)] for index in values)

encoded = torch.tensor(encode(text), dtype=torch.long, device=device)
split_at = int(0.9 * len(encoded))
train_data = encoded[:split_at]
val_data = encoded[split_at:]

block_size = 64
batch_size = 32
print(f"Corpus: {len(text):,} characters; vocabulary: {vocab_size} characters")
print(f"Train: {len(train_data):,} characters; validation: {len(val_data):,} characters")

Corpus: 13,632 characters; vocabulary: 32 characters
Train: 12,268 characters; validation: 1,364 characters


## Model

The attention mask is lower triangular: position `t` can attend only to positions `0` through `t`. The residual paths and layer normalization follow a compact pre-norm Transformer block.

In [17]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("embedding width must be divisible by head count")
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        batch, time, channels = x.shape
        q, k, v = self.qkv(x).view(
            batch, time, 3, self.n_head, self.head_size
        ).unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        causal_mask = torch.ones(time, time, dtype=torch.bool, device=x.device).tril()
        scores = scores.masked_fill(~causal_mask, torch.finfo(scores.dtype).min)
        weights = F.softmax(scores, dim=-1)
        attended = weights @ v
        attended = attended.transpose(1, 2).contiguous().view(batch, time, channels)
        return self.proj(attended)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        return x + self.feed_forward(self.ln2(x))


class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=64, n_head=4, n_layer=2):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *(TransformerBlock(n_embd, n_head) for _ in range(n_layer))
        )
        self.final_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        batch, time = idx.shape
        if time > self.block_size:
            raise ValueError(f"sequence length {time} exceeds block size {self.block_size}")
        positions = torch.arange(time, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.blocks(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            context = idx[:, -self.block_size:]
            logits, _ = self(context)
            next_token = torch.multinomial(F.softmax(logits[:, -1, :], dim=-1), 1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx


model = TinyLanguageModel(vocab_size, block_size).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,}")

Parameters: 108,320


## One forward pass and tensor shapes

A batch has shape `(batch, time)`. The model returns one vocabulary-sized score vector per input character, and the shifted target has one expected next character for each score vector. The scalar loss averages cross-entropy over all batch/time positions.

In [18]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    starts = torch.randint(len(data) - block_size, (batch_size,), device=device)
    offsets = torch.arange(block_size, device=device)
    x = data[starts[:, None] + offsets]
    y = data[starts[:, None] + offsets + 1]
    return x, y


xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print(f"input tokens: {tuple(xb.shape)}   (batch, time)")
print(f"target tokens: {tuple(yb.shape)}   (batch, time; next character)")
print(f"logits: {tuple(logits.shape)}   (batch, time, vocabulary)")
print(f"loss: {loss.item():.4f}   (scalar mean over batch and time)")

input tokens: (32, 64)   (batch, time)
target tokens: (32, 64)   (batch, time; next character)
logits: (32, 64, 32)   (batch, time, vocabulary)
loss: 3.5627   (scalar mean over batch and time)


## Numerical gradient check

Before training changes the weights, compare `backward()` with a central finite difference for one output-head weight. Both perturbed losses use the same fixed batch and evaluation mode, so dropout or new data cannot explain a discrepancy. The estimate is `(loss(w + ε) - loss(w - ε)) / (2ε)`.

In [19]:
model.eval()
model.zero_grad(set_to_none=True)
checked_parameter = model.lm_head.weight
checked_index = (0, 0)
epsilon = 1e-2

_, check_loss = model(xb, yb)
check_loss.backward()
backprop_gradient = checked_parameter.grad[checked_index].item()
original_weight = checked_parameter[checked_index].item()

with torch.no_grad():
    checked_parameter[checked_index] = original_weight + epsilon
    loss_plus = model(xb, yb)[1].item()
    checked_parameter[checked_index] = original_weight - epsilon
    loss_minus = model(xb, yb)[1].item()
    checked_parameter[checked_index] = original_weight

finite_difference = (loss_plus - loss_minus) / (2 * epsilon)
absolute_error = abs(backprop_gradient - finite_difference)
relative_error = absolute_error / max(abs(backprop_gradient), abs(finite_difference), 1e-12)
print(f"parameter: lm_head.weight{checked_index}; epsilon: {epsilon:.3g}")
print(f"base loss:             {check_loss.item():.8f}")
print(f"loss at w + epsilon:   {loss_plus:.8f}")
print(f"loss at w - epsilon:   {loss_minus:.8f}")
print(f"backprop gradient:     {backprop_gradient:.8f}")
print(f"finite-diff gradient:  {finite_difference:.8f}")
print(f"absolute error:        {absolute_error:.8g}")
print(f"relative error:        {relative_error:.8g}")
model.zero_grad(set_to_none=True)
model.train()

parameter: lm_head.weight(0, 0); epsilon: 0.01
base loss:             3.56272149
loss at w + epsilon:   3.56259036
loss at w - epsilon:   3.56285667
backprop gradient:     -0.01331639
finite-diff gradient:  -0.01331568
absolute error:        7.1711838e-07
relative error:        5.3852292e-05


TinyLanguageModel(
  (token_embedding): Embedding(32, 64)
  (position_embedding): Embedding(64, 64)
  (blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attention): CausalSelfAttention(
        (qkv): Linear(in_features=64, out_features=192, bias=True)
        (proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (feed_forward): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
    )
    (1): TransformerBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attention): CausalSelfAttention(
        (qkv): Linear(in_features=64, out_features=192, bias=True)
        (proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, ele

The estimates agree to about `5.7e-6` absolute error (`0.059%` relative error) for this weight and batch. This small difference is expected: the loss and perturbations use finite-precision arithmetic, and a finite step approximates rather than exactly evaluates the local derivative. A much larger discrepancy would warrant checking that the batch, model mode, and parameter were held fixed.

## Baseline training loop

This loop uses AdamW and cross-entropy over every next-character target in each batch. Evaluation switches to inference mode and reports means across multiple randomly sampled batches. The settings are intentionally modest; later sections will use controlled experiments rather than treating this loss curve as a conclusion.

In [20]:
@torch.no_grad()
def estimate_loss(eval_batches=10):
    model.eval()
    estimates = {}
    for split in ("train", "validation"):
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch("train" if split == "train" else "validation")
            _, batch_loss = model(x, y)
            losses.append(batch_loss.item())
        estimates[split] = sum(losses) / len(losses)
    model.train()
    return estimates


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
steps = 500
history = {"step": [], "train_loss": [], "validation_loss": []}

for step in range(steps + 1):
    if step % 50 == 0:
        estimate = estimate_loss()
        history["step"].append(step)
        history["train_loss"].append(estimate["train"])
        history["validation_loss"].append(estimate["validation"])
        print(
            f"step {step:3d} | train loss {estimate['train']:.4f} "
            f"| validation loss {estimate['validation']:.4f}"
        )
    if step == steps:
        break
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step   0 | train loss 3.5595 | validation loss 3.5568
step  50 | train loss 2.0578 | validation loss 2.0488
step 100 | train loss 1.3674 | validation loss 1.3473
step 150 | train loss 0.5750 | validation loss 0.5815
step 200 | train loss 0.2504 | validation loss 0.2603
step 250 | train loss 0.1429 | validation loss 0.1462
step 300 | train loss 0.1120 | validation loss 0.1109
step 350 | train loss 0.0932 | validation loss 0.0948
step 400 | train loss 0.0872 | validation loss 0.0885
step 450 | train loss 0.0765 | validation loss 0.0804
step 500 | train loss 0.0761 | validation loss 0.0769


## Sample

Generation is stochastic and is included as a quick sanity check, not as a quality evaluation. The next steps will add the assignment's gradient, accumulation, norm, utilization, and floating-point measurements.

In [21]:
model.eval()
prompt = torch.tensor([encode("A careful model learns")], dtype=torch.long, device=device)
sample = model.generate(prompt, max_new_tokens=180)
print(decode(sample[0].tolist()))

A careful model learns one character at a time. A useful training loop measures the loss, checks the gradients, updates ean urat eack of. The model chan use the past, but never the futsture.
Smakefull m
